# LoRA Adapter Full Onboarding Cycle

Downloads two public Qwen3-0.6B LoRA adapters, logs them as tracked MLflow runs, and
progresses them through the full registry lifecycle:

```
HuggingFace Hub
  └─ snapshot_download → staging dir
       └─ mlflow.log_artifacts → S3 artifact store  (run created)
            └─ mlflow.register_model → Model Registry
                 └─ set_alias("challenger")
                      └─ [Airflow eval DAG runs, you review metrics]
                           └─ set_alias("champion")
                                └─ docker compose up --force-recreate vllm-adapter-sync vllm
```

| # | HF Repo | Registry Name | Task |
|---|---------|---------------|------|
| A | [phh/Qwen3-0.6B-TLDR-Lora](https://huggingface.co/phh/Qwen3-0.6B-TLDR-Lora) | `lora-summarization` | GRPO — TL;DR summarization |
| B | [yil384/CodeV-R1-Distill-Qwen3-0.6b-Lora-py](https://huggingface.co/yil384/CodeV-R1-Distill-Qwen3-0.6b-Lora-py) | `lora-coding` | R1-distill — Python code generation |

**Prerequisites:** MLflow server running · S3 credentials in `infra/compose/.env` · `huggingface_hub mlflow python-dotenv` installed

---
## 0. Imports

In [6]:
from __future__ import annotations

import os
import shutil
from pathlib import Path

import mlflow
from huggingface_hub import snapshot_download
from mlflow.exceptions import MlflowException
from mlflow.tracking import MlflowClient

---
## 1. Configuration

This notebook is designed to run inside the **Jupyter container** (`http://mlflow:5000`, no auth).

| Scenario | `MLFLOW_TRACKING_URI` |
|---|---|
| **Jupyter container** (default) | `http://mlflow:5000` — injected via `MLFLOW_TRACKING_URI` env var |
| Notebook on the **host server** | `http://127.0.0.1:5050` |

All credentials (`MLFLOW_TRACKING_URI`, `AWS_*`, `MLFLOW_S3_ENDPOINT_URL`) are injected into
the container via `docker-compose.yaml` — no `.env` file loading needed.

In [7]:
PROJECT_ROOT = Path("/home/jovyan")
assert (PROJECT_ROOT / "assets").exists()

In [8]:
os.getcwd()

'/home/jovyan'

In [9]:
MLFLOW_TRACKING_URI = os.environ["MLFLOW_TRACKING_URI"]

In [10]:
EXPERIMENT_NAME = "adapters/pretrained-lora"

# ── Staging dir (temporary, cleaned up after each MLflow upload) ──────────────
# Uses /home/jovyan/assets which is the assets/ volume mount.
STAGING_DIR = PROJECT_ROOT / "assets" / "adapters" / "_hf_stage"
STAGING_DIR.mkdir(parents=True, exist_ok=True)

# ── Adapter A: summarization ──────────────────────────────────────────────────
SUMM_HF_REPO = "phh/Qwen3-0.6B-TLDR-Lora"
SUMM_REGISTRY = "lora-summarization"
SUMM_DESCRIPTION = (
    "Pre-trained TLDR summarization LoRA for Qwen3-0.6B. "
    "Trained with GRPO on trl-lib/tldr. "
    "Source: phh/Qwen3-0.6B-TLDR-Lora (HuggingFace)."
)
SUMM_TAGS = {
    "task": "summarization",
    "base_model": "Qwen/Qwen3-0.6B",
    "source": "huggingface",
    "hf_repo": SUMM_HF_REPO,
    "method": "grpo",
}

# ── Adapter B: coding ─────────────────────────────────────────────────────────
CODE_HF_REPO = "yil384/CodeV-R1-Distill-Qwen3-0.6b-Lora-py"
CODE_REGISTRY = "lora-coding"
CODE_DESCRIPTION = (
    "Pre-trained Python code generation LoRA for Qwen3-0.6B. "
    "Distilled from DeepSeek-R1 reasoning traces (CodeV). "
    "Source: yil384/CodeV-R1-Distill-Qwen3-0.6b-Lora-py (HuggingFace)."
)
CODE_TAGS = {
    "task": "code-generation",
    "base_model": "Qwen/Qwen3-0.6B",
    "source": "huggingface",
    "hf_repo": CODE_HF_REPO,
    "method": "r1-distillation",
}

print(f"MLflow tracking URI : {MLFLOW_TRACKING_URI}")
print(f"Staging dir         : {STAGING_DIR}")
print(f"Experiment          : {EXPERIMENT_NAME}")
print(f"Adapter A           : {SUMM_REGISTRY}  ←  {SUMM_HF_REPO}")
print(f"Adapter B           : {CODE_REGISTRY}  ←  {CODE_HF_REPO}")

MLflow tracking URI : http://mlflow:5000
Staging dir         : /home/jovyan/assets/adapters/_hf_stage
Experiment          : adapters/pretrained-lora
Adapter A           : lora-summarization  ←  phh/Qwen3-0.6B-TLDR-Lora
Adapter B           : lora-coding  ←  yil384/CodeV-R1-Distill-Qwen3-0.6b-Lora-py


---
## 2. Connect to MLflow

In [11]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

client = MlflowClient()

existing = [m.name for m in client.search_registered_models()]

In [12]:
experiment = mlflow.set_experiment(EXPERIMENT_NAME)

print(f"artifact_location: {experiment.artifact_location}")
print(f"creation_time: {experiment.creation_time}")
print(f"experiment_id: {experiment.experiment_id}")
print(f"last_update_time: {experiment.last_update_time}")
print(f"lifecycle_stage: {experiment.lifecycle_stage}")
print(f"name: {experiment.name}")
print(f"tags: {experiment.tags}")
print(f"workspace: {experiment.workspace}")

artifact_location: s3://agent-042-mlflow-artifacts/mlflow/1
creation_time: 1774248418110
experiment_id: 1
last_update_time: 1774248418110
lifecycle_stage: active
name: adapters/pretrained-lora
tags: {}
workspace: default


---
## Part I — Download, Log & Register as Challengers

Each adapter follows four explicit steps:

1. **Download** — `snapshot_download` from HuggingFace to a local staging dir
2. **Log** — `mlflow.log_artifacts` uploads files to S3; a run is created for traceability
3. **Register** — `mlflow.register_model` creates a versioned entry in the Model Registry
4. **Alias** — `set_registered_model_alias("challenger")` marks it as "ready to evaluate"

The **challenger** alias signals "candidate in evaluation, not yet in production."  
Promotion to **champion** happens in Part II after you have reviewed metrics in Airflow.

---
### A · `lora-summarization`

**HuggingFace repo:** [`phh/Qwen3-0.6B-TLDR-Lora`](https://huggingface.co/phh/Qwen3-0.6B-TLDR-Lora)  
**Task:** TL;DR summarization — GRPO-trained on `trl-lib/tldr`

#### Step A-1 · Download from HuggingFace Hub

In [13]:
summ_local_dir = STAGING_DIR / SUMM_HF_REPO.replace("/", "--")

print(f"Downloading {SUMM_HF_REPO} …")
snapshot_download(
    repo_id=SUMM_HF_REPO,
    local_dir=str(summ_local_dir),
    ignore_patterns=["*.bin"],  # prefer safetensors; skip legacy .bin weights
)

files = sorted(f.name for f in summ_local_dir.iterdir())
print(f"✓ Saved to {summ_local_dir}")
print(f"  Files: {files}")

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

✓ Saved to /home/jovyan/assets/adapters/_hf_stage/phh--Qwen3-0.6B-TLDR-Lora
  Files: ['.cache', '.gitattributes', 'Qwen3-0.6B-tldr-lora-f16.gguf', 'README.md', 'adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'merges.txt', 'optimizer.pt', 'rng_state.pth', 'scheduler.pt', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'trainer_state.json', 'vocab.json']


#### Step A-2 · Log adapter files as a tracked MLflow run

In [14]:
with mlflow.start_run(
    run_name=f"{SUMM_REGISTRY}-pretrained",
    tags={**SUMM_TAGS, "registered_by": "register_pretrained_loras.ipynb"},
    description=SUMM_DESCRIPTION,
) as run:
    mlflow.log_param("hf_repo", SUMM_HF_REPO)
    mlflow.log_param("source", "huggingface")
    # Log all adapter files under "model/" — this is the standard path
    # so that  runs:/{run_id}/model  works as the model_uri below.
    mlflow.log_artifacts(str(summ_local_dir), artifact_path="model")
    summ_run_id = run.info.run_id

# Clean up staging immediately after upload
shutil.rmtree(summ_local_dir)

print("✓ Artifacts uploaded to MLflow")
print(f"  run_id  : {summ_run_id}")
print("  Staging : removed")

🏃 View run lora-summarization-pretrained at: http://mlflow:5000/#/experiments/1/runs/83d8f06c99f54ddb8b2c19dfd33ee990
🧪 View experiment at: http://mlflow:5000/#/experiments/1
✓ Artifacts uploaded to MLflow
  run_id  : 83d8f06c99f54ddb8b2c19dfd33ee990
  Staging : removed


#### Step A-3 · Register in Model Registry

In [15]:

# mlflow.register_model() requires a LoggedModel entry (mlflow.<flavor>.log_model),
# but log_artifacts() only stores raw files. Use create_model_version() directly.
try:
    client.create_registered_model(
        name=SUMM_REGISTRY,
        tags=SUMM_TAGS,
        description=SUMM_DESCRIPTION,
    )
except MlflowException as e:
    if e.error_code != "RESOURCE_ALREADY_EXISTS":
        raise

artifact_uri = client.get_run(summ_run_id).info.artifact_uri
summ_mv = client.create_model_version(
    name=SUMM_REGISTRY,
    source=f"{artifact_uri}/model",
    run_id=summ_run_id,
    tags=SUMM_TAGS,
    description=SUMM_DESCRIPTION,
)

print(f"✓ Registered:  {SUMM_REGISTRY}  version {summ_mv.version}")
print(f"  source: {summ_mv.source}")


2026/03/23 07:35:51 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: lora-summarization, version 1


✓ Registered:  lora-summarization  version 1
  source: s3://agent-042-mlflow-artifacts/mlflow/1/83d8f06c99f54ddb8b2c19dfd33ee990/artifacts/model


#### Step A-4 · Assign `challenger` alias

In [16]:
client.set_registered_model_alias(
    name=SUMM_REGISTRY,
    alias="challenger",
    version=str(summ_mv.version),
)

print(f"✓ {SUMM_REGISTRY}  v{summ_mv.version}  →  alias 'challenger'")

✓ lora-summarization  v1  →  alias 'challenger'


---
### B · `lora-coding`

**HuggingFace repo:** [`yil384/CodeV-R1-Distill-Qwen3-0.6b-Lora-py`](https://huggingface.co/yil384/CodeV-R1-Distill-Qwen3-0.6b-Lora-py)  
**Task:** Python code generation — distilled from DeepSeek-R1 reasoning traces

#### Step B-1 · Download from HuggingFace Hub

In [ ]:
code_local_dir = STAGING_DIR / CODE_HF_REPO.replace("/", "--")

print(f"Downloading {CODE_HF_REPO} …")
snapshot_download(
    repo_id=CODE_HF_REPO,
    local_dir=str(code_local_dir),
    ignore_patterns=["*.bin"],
)

files = sorted(f.name for f in code_local_dir.iterdir())
print(f"✓ Saved to {code_local_dir}")
print(f"  Files: {files}")

#### Step B-2 · Log adapter files as a tracked MLflow run

In [ ]:
with mlflow.start_run(
    run_name=f"{CODE_REGISTRY}-pretrained",
    tags={**CODE_TAGS, "registered_by": "register_pretrained_loras.ipynb"},
    description=CODE_DESCRIPTION,
) as run:
    mlflow.log_param("hf_repo", CODE_HF_REPO)
    mlflow.log_param("source", "huggingface")
    mlflow.log_artifacts(str(code_local_dir), artifact_path="model")
    code_run_id = run.info.run_id

shutil.rmtree(code_local_dir)

print("✓ Artifacts uploaded to MLflow")
print(f"  run_id  : {code_run_id}")
print("  Staging : removed")

#### Step B-3 · Register in Model Registry

In [ ]:

try:
    client.create_registered_model(
        name=CODE_REGISTRY,
        tags=CODE_TAGS,
        description=CODE_DESCRIPTION,
    )
except MlflowException as e:
    if e.error_code != "RESOURCE_ALREADY_EXISTS":
        raise

artifact_uri = client.get_run(code_run_id).info.artifact_uri
code_mv = client.create_model_version(
    name=CODE_REGISTRY,
    source=f"{artifact_uri}/model",
    run_id=code_run_id,
    tags=CODE_TAGS,
    description=CODE_DESCRIPTION,
)

print(f"✓ Registered:  {CODE_REGISTRY}  version {code_mv.version}")
print(f"  source: {code_mv.source}")


#### Step B-4 · Assign `challenger` alias

In [ ]:
client.set_registered_model_alias(
    name=CODE_REGISTRY,
    alias="challenger",
    version=str(code_mv.version),
)

print(f"✓ {CODE_REGISTRY}  v{code_mv.version}  →  alias 'challenger'")

---
### Checkpoint — Verify both `challenger` registrations

In [17]:
print(f"{'=' * 65}")
print("  Model Registry — challenger aliases")
print(f"{'=' * 65}")

for name in (SUMM_REGISTRY, CODE_REGISTRY):
    try:
        mv = client.get_model_version_by_alias(name, "challenger")
        print(f"\n  ✓  {name}")
        print(f"       version : {mv.version}")
        print(f"       run_id  : {mv.run_id}")
        print(f"       aliases : {getattr(mv, 'aliases', ['challenger'])}")
    except MlflowException as exc:
        print(f"\n  ✗  {name}  — challenger alias missing: {exc}")

  Model Registry — challenger aliases

  ✓  lora-summarization
       version : 1
       run_id  : 83d8f06c99f54ddb8b2c19dfd33ee990
       aliases : ['challenger']

  ✗  lora-coding  — challenger alias missing: RESOURCE_DOES_NOT_EXIST: Registered Model with name=lora-coding not found


---
## ⏸  Pause — Evaluate in Airflow

Both adapters are now registered as **challengers** in MLflow. Before promoting to production,
review their eval metrics via Airflow:

1. Open Airflow UI (e.g. `http://your-server:8080`)
2. Trigger the **`eval_dags`** pipeline — it queries each model's `challenger` version and
   runs the configured retrieval / generation benchmarks
3. Open the **`adapters/pretrained-lora`** experiment in the MLflow UI and inspect the logged
   eval scores for each run
4. If both adapters meet your quality thresholds, **come back here and run Part II**

> **Note:** If you restart the Jupyter kernel before running Part II, re-run cells 1–3
> (Imports, Configuration, Connect) to restore `client`, `SUMM_REGISTRY`, and `CODE_REGISTRY`
> in the kernel state. The registry data in MLflow is persisted — you do not need to re-run
> Part I.

---
## Part II — Promote Challengers to Champion

Run this section only after Airflow evaluation confirms both adapters meet quality thresholds.

The `adapter-sync` init container and vLLM only track the **`champion`** alias — promoting here
is the gate that puts an adapter into production on the next service restart.

#### Promote `lora-summarization` → champion

In [ ]:
# Re-query from the registry so this cell is safe to run after a kernel restart.
summ_challenger = client.get_model_version_by_alias(SUMM_REGISTRY, "challenger")
print(f"Current challenger:  {SUMM_REGISTRY}  v{summ_challenger.version}")

client.set_registered_model_alias(
    name=SUMM_REGISTRY,
    alias="champion",
    version=str(summ_challenger.version),
)

print(f"✓ {SUMM_REGISTRY}  v{summ_challenger.version}  →  alias 'champion'")

#### Promote `lora-coding` → champion

In [ ]:
code_challenger = client.get_model_version_by_alias(CODE_REGISTRY, "challenger")
print(f"Current challenger:  {CODE_REGISTRY}  v{code_challenger.version}")

client.set_registered_model_alias(
    name=CODE_REGISTRY,
    alias="champion",
    version=str(code_challenger.version),
)

print(f"✓ {CODE_REGISTRY}  v{code_challenger.version}  →  alias 'champion'")

#### Verify champions

In [ ]:
print(f"{'=' * 65}")
print("  Model Registry — champion aliases")
print(f"{'=' * 65}")

all_ok = True
for name in (SUMM_REGISTRY, CODE_REGISTRY):
    try:
        mv = client.get_model_version_by_alias(name, "champion")
        print(f"\n  ✓  {name}")
        print(f"       version : {mv.version}")
        print(f"       run_id  : {mv.run_id}")
        print(f"       source  : {mv.source}")
        print(f"       aliases : {getattr(mv, 'aliases', ['champion'])}")
    except MlflowException as exc:
        print(f"\n  ✗  {name}  — champion alias missing: {exc}")
        all_ok = False

print(f"\n{'=' * 65}")
if all_ok:
    print("Both adapters promoted to champion. Proceed to Part III.")
else:
    print("Some adapters are missing. Re-run the promote cells above.")

---
## Part III — Deploy: Restart vLLM

`adapter-sync` (`vllm-adapter-sync` service) queries every model in the registry with the
`champion` alias, downloads adapter files from S3 to `/adapters/`, and writes `lora-modules.json`.
`vllm-entrypoint.sh` then injects `--lora-modules @/adapters/lora-modules.json` automatically.

### Option A — Full restart (recommended)

```bash
cd infra/compose
docker compose up -d --force-recreate vllm-adapter-sync vllm
```

### Option B — Hot-load without downtime (`VLLM_ALLOW_RUNTIME_LORA_UPDATING=true` is already set)

```bash
# Re-run sync to download fresh champion files to /adapters/:
docker compose run --rm vllm-adapter-sync

# Hot-load each adapter into the running vLLM process:
curl -X POST http://localhost:8000/v1/load_lora_adapter \
  -H "Content-Type: application/json" \
  -d '{"lora_name": "lora-summarization", "lora_path": "/adapters/lora-summarization/v1"}'

curl -X POST http://localhost:8000/v1/load_lora_adapter \
  -H "Content-Type: application/json" \
  -d '{"lora_name": "lora-coding", "lora_path": "/adapters/lora-coding/v1"}'
```

> Replace `v1` with the version numbers printed in the Verify champions cell above.  
> The container path follows: `REGISTRY_ADAPTERS_DIR/{model_name}/v{version}/`

### Verify adapters are loaded

```bash
curl -s http://localhost:8000/v1/models | python3 -m json.tool
```

Both `lora-summarization` and `lora-coding` should appear alongside the base model.

### Unload at runtime (if needed)

```bash
curl -X DELETE http://localhost:8000/v1/unload_lora_adapter \
  -H "Content-Type: application/json" \
  -d '{"lora_name": "lora-coding"}'
```